# Step 4: Build Metadata Labels

Create a clean regression metadata CSV that pairs each galaxy image with a halo-related physical target. The target used here is the log10 transform of a halo mass proxy such as `group_m_crit200`.

This notebook does not create segmentation labels or dark matter location masks.

## 1. Setup Environment

In [ ]:
# Configure Google Colab environment when needed
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    project_path = Path('/content/drive/MyDrive/xai-dark-matter-localization')
    os.chdir(project_path)
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
else:
    project_path = Path.cwd()

print(f'Working directory: {Path.cwd()}')

## 2. Build Regression CSV

The builder validates image paths and physical labels, filters invalid rows, creates train/val/test splits, and saves a compact CSV for regression.

In [ ]:
from pathlib import Path
from src.data.build_regression_dataset import build_regression_metadata

DATASET_DIR = Path('data/processed/TNG-DM-XAI-v1')
input_csv = DATASET_DIR / 'metadata_with_images.csv'
output_csv = DATASET_DIR / 'regression_metadata.csv'

df_regression = build_regression_metadata(
    input_csv=input_csv,
    output_csv=output_csv,
    label_column='group_m_crit200',
    resolution=224,
    project_root=Path.cwd(),
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)

df_regression.head()

## 3. Validate Output Columns

In [ ]:
required_columns = {'image_path', 'subhalo_id', 'halo_mass_raw', 'halo_mass_log10', 'split'}
missing = required_columns - set(df_regression.columns)
assert not missing, f'Missing columns: {missing}'
assert df_regression['image_path'].notna().all()
assert df_regression['halo_mass_raw'].notna().all()
assert (df_regression['halo_mass_raw'] > 0).all()
assert df_regression['split'].isin(['train', 'val', 'test']).all()

print('Regression metadata is valid.')
print(df_regression['split'].value_counts())

## 4. Target Statistics

In [ ]:
target = df_regression['halo_mass_log10']
print('halo_mass_log10 statistics')
print(f'  total: {len(df_regression)}')
print(f'  min:   {target.min():.6f}')
print(f'  max:   {target.max():.6f}')
print(f'  mean:  {target.mean():.6f}')
print(f'  std:   {target.std(ddof=0):.6f}')

## 5. PyTorch Dataset Smoke Check

This opens only one image to verify that the CSV can be consumed lazily. The full dataset is not loaded into memory.

In [ ]:
from src.data.image_dataset import HaloMassImageDataset

train_dataset = HaloMassImageDataset(
    csv_path=output_csv,
    split='train',
    root_dir=Path.cwd(),
    return_metadata=True,
)

print(f'Train samples: {len(train_dataset)}')
if len(train_dataset) > 0:
    image, target, metadata = train_dataset[0]
    print(f'Image tensor/array shape: {getattr(image, "shape", None)}')
    print(f'Target log10 halo mass: {float(target):.6f}')
    print(metadata)

## Expected output

The notebook writes `data/processed/TNG-DM-XAI-v1/regression_metadata.csv` with image paths, subhalo IDs, raw halo masses, log10 halo-mass targets, and train/val/test splits. It also prints missing-data checks and target statistics.